# Virtual Counselor — Algorithmic Prompt Search Demo
**WSU CPTS 440 AI Project** | Jaime Gudino, Andrew Edson, Joseph Buchanan

This notebook demonstrates the full project pipeline:

| Section | What it shows | API key needed? |
|---|---|---|
| 1. Setup | Clone repo & install deps | No |
| 2. Prompt Templates | PromptTemplate + 10 mutation operators | No |
| 3. Beam Search | Search over mutation space (mock LLM) | No |
| 4. MCTS | UCB1-guided tree search (mock LLM) | No |
| 5. RAG Evaluation | Pre-recorded 120-case results + charts | No |
| 6. Live Query *(optional)* | Real Claude Haiku answer via your own key | Yes (your key) |

> **Sections 1–5 run completely offline — no API key required.**

---
## Section 1 · Setup
Clone the repository and install dependencies.

In [ ]:
!git clone https://github.com/gudino27/VC-n8n-440.git
%cd VC-n8n-440/prompt-search
print(' project downloaded')

[Errno 2] No such file or directory: 'VC-n8n-440/prompt-search'
/Volumes/asus/VC-n8n-440/prompt-search/notebooks
 project downloaded


In [1]:

# Install only the packages needed for this demo (no llama-cpp, no torch)
!pip install -q \
    anthropic>=0.40.0 \
    sentence-transformers>=2.7 \
    faiss-cpu>=1.8 \
    pyyaml>=6.0 \
    python-dotenv>=1.0 \
    requests>=2.31 \
    matplotlib>=3.7 \
    numpy>=1.24

print('Setup complete.')

zsh:1: 0.40.0 not found
Setup complete.


In [2]:
import sys, os
sys.path.insert(0, 'src')
print('Python path configured. src/ modules are importable.')

Python path configured. src/ modules are importable.


---
## Section 2 · Prompt Template System

Prompts are represented as structured `PromptTemplate` objects with six mutable fields.  
A `PromptMutator` provides **10 atomic operators** (4 active in search runs) that clone and modify templates while recording the full derivation path.

In [3]:
from prompts.template import PromptTemplate
from prompts.mutations import PromptMutator

# --- Base template ---
base = PromptTemplate(
    system_role="a WSU academic advisor",
    task_description=(
        "Answer the student's question about courses, prerequisites, "
        "or degree requirements accurately and concisely."
    ),
    constraints=["Use WSU course codes", "Cite prerequisites explicitly"],
)

SAMPLE_Q = "Can I take CPTS 360 after completing CPTS 223 and CPTS 260?"

print("=" * 60)
print("BASE TEMPLATE  (mutation path:", base.mutation_path(), ")")
print("=" * 60)
print(base.render(SAMPLE_Q))

ModuleNotFoundError: No module named 'prompts'

In [ ]:
# Apply all 4 active mutations in sequence and inspect the derivation path
mutations = [
    ("add_cot",              PromptMutator.add_cot),
    ("add_verification",     PromptMutator.add_verification_step),
    ("add_self_consistency", PromptMutator.add_self_consistency),
    ("remove_cot",           PromptMutator.remove_cot),
]

template = base
for name, fn in mutations:
    template = fn(template)
    print(f"After {name:<25} path: {template.mutation_path()}")

print()
print("=" * 60)
print("FINAL TEMPLATE after 4 mutations")
print("=" * 60)
print(template.render(SAMPLE_Q))

---
## Section 3 · Beam Search over Prompt Mutation Space

**Algorithm:** At each iteration every beam candidate is expanded by applying all mutation operators. Candidates are deduplicated by their mutation path, scored on a validation set, and pruned to the top-B.

$$\mathcal{C}_t = \{m(\pi) \mid \pi \in \mathcal{B}_{t-1},\; m \in \mathcal{M}\}, \quad \mathcal{B}_t = \operatorname{top}_B(f(\cdot, \mathcal{D}))(\mathcal{C}_t)$$

A **mock LLM** is used here so no API key is required. It returns pre-recorded answers for the 5 sample questions. The real accuracy results (120 cases, Claude Haiku) are in Section 5.

In [ ]:
from llm.base import BaseLLMClient

class MockLLMClient(BaseLLMClient):
    """Deterministic stub — no API key needed.
    Returns pre-recorded answers. Simulates prompt sensitivity:
    structured prompts (persona + CoT) unlock more correct answers."""

    # (question keyword, vague answer, specific correct answer)
    _QA = [
        ("cpts 122",
         "You may be eligible.",
         "Yes. CPTS 122 requires CPTS 121 with a C or better."),
        ("cpts 360",
         "Check with your advisor.",
         "Yes. Both CPTS 223 and CPTS 260 are satisfied."),
        ("78 credits",
         "You have completed many credits.",
         "42 credits remain for the CS BS (120 total)."),
        ("ucore",
         "Some categories may remain.",
         "Remaining: SSCI, HUM, ARTS, DIVR, INTG/CAPS."),
        ("cpts 451",
         "Verify prerequisites before enrolling.",
         "Yes. CPTS 355 satisfies the CPTS 451 prerequisite."),
    ]

    def __init__(self):
        self._calls = 0

    def generate(self, prompt: str, temperature=0.0, max_tokens=500) -> str:
        self._calls += 1
        p = prompt.lower()
        # A well-structured prompt (has persona AND CoT) unlocks specific answers
        structured = ("academic advisor" in p) and (
            "step by step" in p or "think through" in p
        )
        for keyword, vague, specific in self._QA:
            if keyword in p:
                return specific if structured else vague
        return "Please consult the WSU course catalog."

    def get_usage_stats(self):
        return {"input_tokens": self._calls * 180, "output_tokens": self._calls * 40}


# 5-question validation set with expected answer substrings
MINI_VAL = [
    {"question": "Can I take CPTS 122 if I have completed CPTS 121?",
     "answer": "Yes"},
    {"question": "Can I take CPTS 360 after completing CPTS 223 and CPTS 260?",
     "answer": "Yes"},
    {"question": "How many credits remain if I have 78 credits?",
     "answer": "42"},
    {"question": "I have WRTG, QUAN, COMM, BSCI, PSCI — what UCORE is left?",
     "answer": "SSCI"},
    {"question": "Can I take CPTS 451 after finishing CPTS 355?",
     "answer": "Yes"},
]

print("MockLLMClient ready. Validation set: 5 questions.")
print("Base template (no persona, no CoT) — expected accuracy: ~40%")
print("After mutations (persona + CoT) — expected accuracy: ~100%")

In [ ]:
from search.beam_search import BeamSearchPromptOptimizer

mock = MockLLMClient()

optimizer = BeamSearchPromptOptimizer(
    beam_width=3,
    max_iterations=10,
    llm_client=mock,
    patience=3,
)

# Start from a minimal base (no persona — mock gives vague answers)
base_no_persona = PromptTemplate(
    task_description="Answer the student's advising question.",
)

print("Running Beam Search  (B=3, I_max=10, 4 active mutations)...")
best = optimizer.search(base_no_persona, MINI_VAL)

print(f"\nSearch complete — {mock._calls} mock LLM calls made.")
print(f"Best template mutation path: {best.mutation_path()}")
print("\nIteration history:")
print(f"{'Iter':>5}  {'Best acc':>10}  Path")
for h in optimizer.history:
    print(f"{h['iteration']:>5}  {h['best_accuracy']:>10.2%}  {h['best_path']}")

---
## Section 4 · Monte Carlo Tree Search (MCTS)

**Algorithm:** Each iteration runs four phases — Selection (UCB1), Expansion (random mutation), Simulation (evaluate on validation set), Backpropagation (update ancestors).

$$\text{UCB1}(v) = \frac{w_v}{n_v} + c\sqrt{\frac{\ln n_{\text{parent}}}{n_v}}, \quad c = \sqrt{2} \approx 1.414$$

In [ ]:
from search.mcts import MCTSPromptSearch
import math

mock2 = MockLLMClient()

mcts = MCTSPromptSearch(
    llm_client=mock2,
    exploration_weight=math.sqrt(2),  # c = √2 ≈ 1.414
)

print("Running MCTS  (N=100 iterations, c=√2)...")
best_mcts = mcts.search(base_no_persona, MINI_VAL, num_iterations=100)

print(f"\nSearch complete — {mock2._calls} mock LLM calls made.")
print(f"Best template: {best_mcts.mutation_path()}")
print(f"Root children explored: {len(mcts.root.children)}")

# Show top 5 children by average reward
scored = [
    (c.total_reward / c.visits, c.visits, c.template.mutation_path())
    for c in mcts.root.children if c.visits > 0
]
scored.sort(reverse=True)
print("\nTop 5 nodes by avg reward:")
print(f"{'Avg reward':>12}  {'Visits':>7}  Path")
for reward, visits, path in scored[:5]:
    print(f"{reward:>12.2%}  {visits:>7}  {path}")

---
## Section 5 · RAG Evaluation Results (Pre-recorded)

These results come from the full 120-case evaluation run against **Claude Haiku** (`claude-haiku-4-5`) on 2026-04-27.  
Accuracy = fraction of answers with cosine similarity ≥ 0.60 against ground-truth embeddings (`all-MiniLM-L6-v2`).

In [ ]:
import json, os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Load the authoritative results file
results_path = "data/results/rag_eval_results.json"
with open(results_path) as f:
    results = json.load(f)

rag_overall   = results["rag"]["accuracy"]
norag_overall = results["no_rag"]["accuracy"]
by_cat        = results["by_category"]

# Category display names and counts
CAT_META = {
    "prerequisite_validation":      ("Prereq Validation",    30),
    "prerequisite_chain_discovery": ("Chain Discovery",      25),
    "schedule_feasibility":         ("Schedule Feasibility", 15),
    "ucore_planning":               ("UCORE Planning",       10),
    "credit_calculations":          ("Credit Calculations",  20),
    "degree_progress":              ("Degree Progress",      20),
}

# Sort by RAG accuracy descending (matches the paper table)
by_cat_sorted = sorted(by_cat, key=lambda x: x["rag_accuracy"], reverse=True)

print(f"{'Category':<28} {'n':>4}  {'RAG':>7}  {'No-RAG':>8}  {'Delta':>7}")
print("-" * 62)
for row in by_cat_sorted:
    label, n = CAT_META[row["category"]]
    print(f"{label:<28} {n:>4}  {row['rag_accuracy']:>7.2%}  "
          f"{row['no_rag_accuracy']:>8.2%}  {row['delta']:>+7.2%}")
print("-" * 62)
print(f"{'Overall':<28} {'120':>4}  {rag_overall:>7.2%}  {norag_overall:>8.2%}  "
      f"{rag_overall - norag_overall:>+7.2%}")

In [ ]:
# ── Grouped bar chart: RAG vs No-RAG by category ──────────────────────
labels = [CAT_META[r["category"]][0] for r in by_cat_sorted]
rag_vals   = [r["rag_accuracy"]    for r in by_cat_sorted]
norag_vals = [r["no_rag_accuracy"] for r in by_cat_sorted]

x   = np.arange(len(labels))
w   = 0.35
fig, ax = plt.subplots(figsize=(11, 5))

bars_rag   = ax.bar(x - w/2, rag_vals,   w, label="With RAG",    color="#2196F3", zorder=3)
bars_norag = ax.bar(x + w/2, norag_vals, w, label="Without RAG", color="#FF7043", zorder=3)

ax.axhline(rag_overall,   color="#1565C0", linestyle="--", linewidth=1.2,
           label=f"RAG overall {rag_overall:.1%}")
ax.axhline(norag_overall, color="#BF360C", linestyle="--", linewidth=1.2,
           label=f"No-RAG overall {norag_overall:.1%}")

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Semantic Similarity Accuracy")
ax.set_title("RAG vs. No-RAG Accuracy — 120-case WSU Domain Test Suite\n"
             "(cosine ≥ 0.60, all-MiniLM-L6-v2, Claude Haiku, 2026-04-27)")
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.grid(axis="y", alpha=0.4, zorder=0)
ax.legend(fontsize=8)

for bar in bars_rag:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.0%}", ha="center", va="bottom", fontsize=7)
for bar in bars_norag:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.0%}", ha="center", va="bottom", fontsize=7)

plt.tight_layout()
plt.savefig("rag_results_chart.png", dpi=150)
plt.show()
print("Chart saved to rag_results_chart.png")

In [ ]:
# ── Delta (improvement) bar chart ─────────────────────────────────────
deltas = [r["delta"] for r in by_cat_sorted]
colors = ["#43A047" if d >= 0.5 else "#FDD835" if d >= 0.25 else "#EF5350" for d in deltas]

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(labels, deltas, color=colors, zorder=3)
ax.axhline(rag_overall - norag_overall, color="black", linestyle="--", linewidth=1,
           label=f"Overall Δ = {rag_overall - norag_overall:+.2%}")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("RAG improvement (percentage points)")
ax.set_title("RAG Gain by Category")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:+.0%}"))
ax.grid(axis="y", alpha=0.4, zorder=0)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

---
## Section 6 · Live Query *(Optional — your own API key)*

**To run this section:**
1. Click the 🔑 **Secrets** icon in the Colab left sidebar.
2. Add a secret named `ANTHROPIC_API_KEY` with your key.
3. Enable notebook access for that secret.
4. Run the cells below.

> Your key stays in Colab's secrets manager — it is never stored in this notebook.

In [ ]:
# ── Load API key from Colab Secrets ───────────────────────────────────
try:
    from google.colab import userdata
    CLAUDE_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    print('Key loaded from Colab Secrets.')
except Exception:
    import getpass
    CLAUDE_API_KEY = getpass.getpass('Enter your Anthropic API key: ')

if not CLAUDE_API_KEY:
    raise ValueError('No API key found — skipping live query.')

os.environ['CLAUDE_API_KEY'] = CLAUDE_API_KEY
print('API key set. Ready for live queries.')


In [ ]:
# ── Build the FAISS index (one-time, ~30 seconds) ─────────────────────
import subprocess
print("Building FAISS index from course catalog...")
result = subprocess.run(
    [sys.executable, "scripts/build_index.py"],
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else "(no stdout)")
if result.returncode != 0:
    print("STDERR:", result.stderr[-300:])
else:
    print("Index built successfully.")

In [ ]:
# ── Build demo SQLite DB (catalog, degree plan, Fall 2025 sections) ───
print('Building courses.db from metadata.json...')
result = subprocess.run(
    [sys.executable, 'scripts/build_demo_db.py'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-300:])
else:
    print('DB ready.')

In [ ]:
# ── Run live advising queries through the full RAG pipeline ───────────
from counselor.pipeline import VirtualCounselorPipeline

pipeline = VirtualCounselorPipeline(
    index_dir='data/domain',
    api_key=CLAUDE_API_KEY,
)

QUESTIONS = [
    'What courses do I need to take before CPTS 360?',
    'How many credits remain if I have completed 90 credits toward the CS BS?',
    'Which UCORE categories are satisfied by CPTS 483?',
]

for q in QUESTIONS:
    print('\n' + '=' * 60)
    print('Q:', q)
    print('-' * 60)
    answer = pipeline.ask(q)
    print('A:', answer)
